In [17]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from scipy import stats
from torch.utils.data import TensorDataset, DataLoader

# --- CONFIGURATION OPTIMISÉE ---
CONFIG = {
    'layer_name': 'layer4',
    'n_random_sets': 15,
    'alpha': 0.01,
    'batch_size': 16,         
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# --- 1. PRÉPARATION DU MODÈLE ET HOOKS  ---
class ModelWrapper:
    def __init__(self, model, layer_name):
        self.model = model.to(CONFIG['device'])
        self.model.eval()
        self.layer_name = layer_name
        self.activations = {'value': None} # Initialiser à None
        self.gradients = {'value': None}
        self.hooks = [] # Pour pouvoir les supprimer proprement
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations['value'] = output
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients['value'] = grad_output[0]

        found = False
        for name, module in self.model.named_modules():
            if name == self.layer_name:
                # On stocke les handles pour pouvoir les retirer si besoin
                self.hooks.append(module.register_forward_hook(forward_hook))
                self.hooks.append(module.register_full_backward_hook(backward_hook))
                print(f"Hook attaché à la couche: {name}")
                found = True
                break
        if not found:
            raise ValueError(f"Couche {self.layer_name} introuvable.")

    def get_activation(self, inputs):
        """Extraction optimisée avec nettoyage mémoire immédiat."""
        dataset = TensorDataset(inputs)
        loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=False)
        
        all_acts = []
        
        with torch.no_grad():
            for batch in loader:
                batch_imgs = batch[0].to(CONFIG['device'])
                _ = self.model(batch_imgs)
                
                # Récupération
                act = self.activations['value']
                
                # Global Average Pooling
                if len(act.shape) == 4:
                    act = torch.mean(act, dim=[2, 3])
                
                # Transfert CPU
                all_acts.append(act.detach().cpu().numpy())
                
                # --- CRUCIAL: LIBÉRATION MÉMOIRE GPU IMMÉDIATE ---
                self.activations['value'] = None 
                del batch_imgs, act
        
        # Nettoyage final du cache GPU après la boucle
        torch.cuda.empty_cache()
        return np.concatenate(all_acts)

    def get_gradient(self, img_tensor, target_class_index):
        img_tensor = img_tensor.to(CONFIG['device'])
        
        self.model.zero_grad()
        outputs = self.model(img_tensor)
        
        one_hot = torch.zeros_like(outputs)
        one_hot[0, target_class_index] = 1
        
        outputs.backward(gradient=one_hot, retain_graph=True)
        
        grad = self.gradients['value']
        if len(grad.shape) == 4:
            grad = torch.mean(grad, dim=[2, 3])
            
        grad_cpu = grad.detach().cpu().numpy()
        
        # Nettoyage
        self.gradients['value'] = None
        del img_tensor, outputs, one_hot, grad
        # Pas de empty_cache() ici pour la vitesse (c'est image par image)
        
        return grad_cpu

    def clear_hooks(self):
        """Supprime les hooks pour éviter les fuites si on réinstancie."""
        for h in self.hooks:
            h.remove()
        self.hooks = []


# --- 2. GESTION DES DONNÉES ---
def load_images_from_folder(folder, max_imgs=200):
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    images = []
    if not os.path.exists(folder):
        print(f"Attention: Dossier introuvable {folder}")
        return None
        
    for filename in os.listdir(folder)[:max_imgs]:
        img_path = os.path.join(folder, filename)
        try:
            with Image.open(img_path) as img: # Utilisation de 'with' pour fermer le fichier
                img = img.convert('RGB')
                images.append(preprocess(img))
        except Exception:
            continue
            
    if not images:
        raise ValueError(f"Dossier vide: {folder}")
        
    return torch.stack(images) # Reste sur CPU


# --- 3. ENTRAÎNEMENT DU CAV (Identique) ---
class CAV:
    def __init__(self, concept_acts, random_acts):
        self.concept_acts = concept_acts
        self.random_acts = random_acts
        self.coef = None
        self.accuracy = 0
        self._train()

    def _train(self):
        min_len = min(len(self.concept_acts), len(self.random_acts))
        # Équilibrage simple pour éviter les biais de taille
        c_acts = self.concept_acts[:min_len]
        r_acts = self.random_acts[:min_len]
        
        X = np.concatenate([c_acts, r_acts])
        y = np.concatenate([np.ones(len(c_acts)), np.zeros(len(r_acts))])
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)
        
        # Alpha augmenté légèrement pour plus de robustesse
        lm = SGDClassifier(alpha=0.01, max_iter=1000, tol=1e-3, loss='hinge')
        lm.fit(X_train, y_train)
        
        self.coef = lm.coef_[0]
        self.accuracy = lm.score(X_test, y_test)


# --- 4. CALCUL DU SCORE TCAV (Identique) ---
def compute_tcav_score(model_wrapper, target_imgs, cav_vec, target_class_idx):
    sensitivities = []
    # Batch de 1 pour le calcul de gradient (limite mémoire)
    for i in range(len(target_imgs)):
        img_tensor = target_imgs[i].unsqueeze(0)
        grad = model_wrapper.get_gradient(img_tensor, target_class_idx)
        
        cav_norm = cav_vec / np.linalg.norm(cav_vec)
        sens = np.dot(grad.flatten(), cav_norm.flatten())
        sensitivities.append(sens)
    
    sensitivities = np.array(sensitivities)
    score_fraction = np.sum(sensitivities > 0) / len(sensitivities)
    
    pos_magnitude = np.sum(np.abs(sensitivities[sensitivities > 0]))
    total_magnitude = np.sum(np.abs(sensitivities))
    score_magnitude = pos_magnitude / total_magnitude if total_magnitude > 0 else 0.0
    
    return score_fraction, score_magnitude


# --- 5. PIPELINE PRINCIPAL (CORRIGÉ AVEC NETTOYAGE) ---
def run_tcav_analysis(concept_path, target_path, random_root_path, target_class_idx):
    # 1. Grand nettoyage avant de commencer (Pour Kaggle)
    torch.cuda.empty_cache()
    gc.collect()
    
    # 2. Chargement Modèle
    print("--- Chargement du modèle ResNet50 ---")
    # Utilisation de 'weights' au lieu de 'pretrained' (deprecated)
    weights = models.ResNet50_Weights.IMAGENET1K_V1
    model = models.resnet50(weights=weights)
    wrapper = ModelWrapper(model, CONFIG['layer_name'])
    
    try:
        print("--- 1. Extraction des activations du Concept ---")
        concept_imgs = load_images_from_folder(concept_path)
        concept_acts = wrapper.get_activation(concept_imgs)
        # On peut supprimer les images pixels pour libérer la RAM CPU si on est limite
        del concept_imgs
        gc.collect()
        
        print("--- 2. Extraction des activations Cible ---")
        target_imgs = load_images_from_folder(target_path)
        
        tcav_scores_mag = []
        
        print(f"--- 3. Boucle sur {CONFIG['n_random_sets']} ensembles aléatoires ---")
        for i in range(CONFIG['n_random_sets']):
            random_folder = os.path.join(random_root_path, f"random_{i}")
            
            # Gestion d'erreur si un dossier manque
            try:
                random_imgs = load_images_from_folder(random_folder)
            except ValueError:
                continue

            random_acts = wrapper.get_activation(random_imgs)
            del random_imgs # Libère RAM CPU immédiatement
            
            cav = CAV(concept_acts, random_acts)
            
            if cav.accuracy < 0.65: # Seuil légèrement baissé
                print(f"Warning: CAV faible pour Random_{i} (Acc: {cav.accuracy:.2f})")
                
            _, score_mag = compute_tcav_score(wrapper, target_imgs, cav.coef, target_class_idx)
            tcav_scores_mag.append(score_mag)
            print(f"Run {i}: Score Mag = {score_mag:.4f}")
            
            # Nettoyage intermédiaire
            del random_acts, cav
            gc.collect()

        # --- 4. Test Statistique ---
        if len(tcav_scores_mag) > 1:
            avg_score = np.mean(tcav_scores_mag)
            std_score = np.std(tcav_scores_mag)
            t_stat, p_val = stats.ttest_1samp(tcav_scores_mag, 0.5)
            
            print("\n--- RÉSULTATS FINAUX ---")
            print(f"Score TCAV Moyen : {avg_score:.4f} ± {std_score:.4f}")
            print(f"P-value : {p_val:.5e}")
            
            if p_val < 0.05:
                interpretation = "POSITIF" if avg_score > 0.5 else "NÉGATIF"
                print(f"RÉSULTAT: Significatif ({interpretation})")
            else:
                print("RÉSULTAT: Non significatif (Bruit)")
        else:
            print("Pas assez de scores calculés.")

    finally:
        # Nettoyage final pour ne pas laisser le GPU encombré
        wrapper.clear_hooks()
        del wrapper, model
        torch.cuda.empty_cache()
        gc.collect()
        print("--- Mémoire GPU libérée ---")

# --- EXECUTION ---
if __name__ == "__main__":
    # Index ImageNet pour 'Zebra' est 340
    # Assurez-vous que les chemins sont corrects
    run_tcav_analysis(
        '/kaggle/input/data-tcav/data/concepts/stripes', 
        '/kaggle/input/data-tcav/data/target/zebra', 
        '/kaggle/input/data-tcav/data/random', 
        340
    )

--- Chargement du modèle ResNet50 ---
Hook attaché à la couche: layer4
--- 1. Extraction des activations du Concept ---
--- 2. Extraction des activations Cible ---
--- 3. Boucle sur 15 ensembles aléatoires ---


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:829: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:179.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Run 0: Score Mag = 1.0000
Run 1: Score Mag = 1.0000
Run 2: Score Mag = 1.0000
Run 3: Score Mag = 1.0000
Run 4: Score Mag = 1.0000
Run 5: Score Mag = 1.0000
Run 6: Score Mag = 1.0000
Run 7: Score Mag = 1.0000
Run 8: Score Mag = 1.0000
Run 9: Score Mag = 1.0000
Run 10: Score Mag = 1.0000
Run 11: Score Mag = 1.0000
Run 12: Score Mag = 1.0000
Run 13: Score Mag = 1.0000
Run 14: Score Mag = 1.0000

--- RÉSULTATS FINAUX ---
Score TCAV Moyen : 1.0000 ± 0.0000
P-value : 0.00000e+00
✅ RÉSULTAT: Significatif (POSITIF)
--- Mémoire GPU libérée ---


/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [18]:
# --- EXECUTION ---
if __name__ == "__main__":
    # Index ImageNet pour 'Zebra' est 340
    # Assurez-vous que les chemins sont corrects
    run_tcav_analysis(
        '/kaggle/input/datasets/younessouarda/lion-300/lion_300/concept_grass', 
        '/kaggle/input/data-tcav/data/target/zebra', 
        '/kaggle/input/data-tcav/data/random', 
        340
    )

--- Chargement du modèle ResNet50 ---
Hook attaché à la couche: layer4
--- 1. Extraction des activations du Concept ---
--- 2. Extraction des activations Cible ---
--- 3. Boucle sur 15 ensembles aléatoires ---
Run 0: Score Mag = 1.0000
Run 1: Score Mag = 1.0000
Run 2: Score Mag = 0.0000
Run 3: Score Mag = 1.0000
Run 4: Score Mag = 1.0000
Run 5: Score Mag = 1.0000
Run 6: Score Mag = 0.0000
Run 7: Score Mag = 1.0000
Run 8: Score Mag = 0.0000
Run 9: Score Mag = 1.0000
Run 10: Score Mag = 0.0000
Run 11: Score Mag = 1.0000
Run 12: Score Mag = 1.0000
Run 13: Score Mag = 1.0000
Run 14: Score Mag = 1.0000

--- RÉSULTATS FINAUX ---
Score TCAV Moyen : 0.7333 ± 0.4422
P-value : 6.84172e-02
❌ RÉSULTAT: Non significatif (Bruit)
--- Mémoire GPU libérée ---


In [19]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from scipy import stats
from torch.utils.data import TensorDataset, DataLoader

# --- CONFIGURATION OPTIMISÉE ---
CONFIG = {
    'layer_name': 'layer3',
    'n_random_sets': 15,
    'alpha': 0.01,
    'batch_size': 16,         # <--- RÉDUIT À 16 POUR LA SÉCURITÉ MÉMOIRE
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# --- 1. PRÉPARATION DU MODÈLE ET HOOKS (CORRIGÉ) ---
class ModelWrapper:
    def __init__(self, model, layer_name):
        self.model = model.to(CONFIG['device'])
        self.model.eval()
        self.layer_name = layer_name
        self.activations = {'value': None} # Initialiser à None
        self.gradients = {'value': None}
        self.hooks = [] # Pour pouvoir les supprimer proprement
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations['value'] = output
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients['value'] = grad_output[0]

        found = False
        for name, module in self.model.named_modules():
            if name == self.layer_name:
                # On stocke les handles pour pouvoir les retirer si besoin
                self.hooks.append(module.register_forward_hook(forward_hook))
                self.hooks.append(module.register_full_backward_hook(backward_hook))
                print(f"Hook attaché à la couche: {name}")
                found = True
                break
        if not found:
            raise ValueError(f"Couche {self.layer_name} introuvable.")

    def get_activation(self, inputs):
        """Extraction optimisée avec nettoyage mémoire immédiat."""
        dataset = TensorDataset(inputs)
        loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=False)
        
        all_acts = []
        
        with torch.no_grad():
            for batch in loader:
                batch_imgs = batch[0].to(CONFIG['device'])
                _ = self.model(batch_imgs)
                
                # Récupération
                act = self.activations['value']
                
                # Global Average Pooling
                if len(act.shape) == 4:
                    act = torch.mean(act, dim=[2, 3])
                
                # Transfert CPU
                all_acts.append(act.detach().cpu().numpy())
                
                # --- CRUCIAL: LIBÉRATION MÉMOIRE GPU IMMÉDIATE ---
                self.activations['value'] = None 
                del batch_imgs, act
        
        # Nettoyage final du cache GPU après la boucle
        torch.cuda.empty_cache()
        return np.concatenate(all_acts)

    def get_gradient(self, img_tensor, target_class_index):
        img_tensor = img_tensor.to(CONFIG['device'])
        
        self.model.zero_grad()
        outputs = self.model(img_tensor)
        
        one_hot = torch.zeros_like(outputs)
        one_hot[0, target_class_index] = 1
        
        outputs.backward(gradient=one_hot, retain_graph=True)
        
        grad = self.gradients['value']
        if len(grad.shape) == 4:
            grad = torch.mean(grad, dim=[2, 3])
            
        grad_cpu = grad.detach().cpu().numpy()
        
        # Nettoyage
        self.gradients['value'] = None
        del img_tensor, outputs, one_hot, grad
        # Pas de empty_cache() ici pour la vitesse (c'est image par image)
        
        return grad_cpu

    def clear_hooks(self):
        """Supprime les hooks pour éviter les fuites si on réinstancie."""
        for h in self.hooks:
            h.remove()
        self.hooks = []


# --- 2. GESTION DES DONNÉES ---
def load_images_from_folder(folder, max_imgs=200):
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    images = []
    if not os.path.exists(folder):
        print(f"Attention: Dossier introuvable {folder}")
        return None
        
    for filename in os.listdir(folder)[:max_imgs]:
        img_path = os.path.join(folder, filename)
        try:
            with Image.open(img_path) as img: # Utilisation de 'with' pour fermer le fichier
                img = img.convert('RGB')
                images.append(preprocess(img))
        except Exception:
            continue
            
    if not images:
        raise ValueError(f"Dossier vide: {folder}")
        
    return torch.stack(images) # Reste sur CPU


# --- 3. ENTRAÎNEMENT DU CAV (Identique) ---
class CAV:
    def __init__(self, concept_acts, random_acts):
        self.concept_acts = concept_acts
        self.random_acts = random_acts
        self.coef = None
        self.accuracy = 0
        self._train()

    def _train(self):
        min_len = min(len(self.concept_acts), len(self.random_acts))
        # Équilibrage simple pour éviter les biais de taille
        c_acts = self.concept_acts[:min_len]
        r_acts = self.random_acts[:min_len]
        
        X = np.concatenate([c_acts, r_acts])
        y = np.concatenate([np.ones(len(c_acts)), np.zeros(len(r_acts))])
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)
        
        # Alpha augmenté légèrement pour plus de robustesse
        lm = SGDClassifier(alpha=0.01, max_iter=1000, tol=1e-3, loss='hinge')
        lm.fit(X_train, y_train)
        
        self.coef = lm.coef_[0]
        self.accuracy = lm.score(X_test, y_test)


# --- 4. CALCUL DU SCORE TCAV (Identique) ---
def compute_tcav_score(model_wrapper, target_imgs, cav_vec, target_class_idx):
    sensitivities = []
    # Batch de 1 pour le calcul de gradient (limite mémoire)
    for i in range(len(target_imgs)):
        img_tensor = target_imgs[i].unsqueeze(0)
        grad = model_wrapper.get_gradient(img_tensor, target_class_idx)
        
        cav_norm = cav_vec / np.linalg.norm(cav_vec)
        sens = np.dot(grad.flatten(), cav_norm.flatten())
        sensitivities.append(sens)
    
    sensitivities = np.array(sensitivities)
    score_fraction = np.sum(sensitivities > 0) / len(sensitivities)
    
    pos_magnitude = np.sum(np.abs(sensitivities[sensitivities > 0]))
    total_magnitude = np.sum(np.abs(sensitivities))
    score_magnitude = pos_magnitude / total_magnitude if total_magnitude > 0 else 0.0
    
    return score_fraction, score_magnitude


# --- 5. PIPELINE PRINCIPAL (CORRIGÉ AVEC NETTOYAGE) ---
def run_tcav_analysis(concept_path, target_path, random_root_path, target_class_idx):
    # 1. Grand nettoyage avant de commencer (Pour Kaggle)
    torch.cuda.empty_cache()
    gc.collect()
    
    # 2. Chargement Modèle
    print("--- Chargement du modèle ResNet50 ---")
    # Utilisation de 'weights' au lieu de 'pretrained' (deprecated)
    weights = models.ResNet50_Weights.IMAGENET1K_V1
    model = models.resnet50(weights=weights)
    wrapper = ModelWrapper(model, CONFIG['layer_name'])
    
    try:
        print("--- 1. Extraction des activations du Concept ---")
        concept_imgs = load_images_from_folder(concept_path)
        concept_acts = wrapper.get_activation(concept_imgs)
        # On peut supprimer les images pixels pour libérer la RAM CPU si on est limite
        del concept_imgs
        gc.collect()
        
        print("--- 2. Extraction des activations Cible ---")
        target_imgs = load_images_from_folder(target_path)
        
        tcav_scores_mag = []
        
        print(f"--- 3. Boucle sur {CONFIG['n_random_sets']} ensembles aléatoires ---")
        for i in range(CONFIG['n_random_sets']):
            random_folder = os.path.join(random_root_path, f"random_{i}")
            
            # Gestion d'erreur si un dossier manque
            try:
                random_imgs = load_images_from_folder(random_folder)
            except ValueError:
                continue

            random_acts = wrapper.get_activation(random_imgs)
            del random_imgs # Libère RAM CPU immédiatement
            
            cav = CAV(concept_acts, random_acts)
            
            if cav.accuracy < 0.65: # Seuil légèrement baissé
                print(f"Warning: CAV faible pour Random_{i} (Acc: {cav.accuracy:.2f})")
                
            _, score_mag = compute_tcav_score(wrapper, target_imgs, cav.coef, target_class_idx)
            tcav_scores_mag.append(score_mag)
            print(f"Run {i}: Score Mag = {score_mag:.4f}")
            
            # Nettoyage intermédiaire
            del random_acts, cav
            gc.collect()

        # --- 4. Test Statistique ---
        if len(tcav_scores_mag) > 1:
            avg_score = np.mean(tcav_scores_mag)
            std_score = np.std(tcav_scores_mag)
            t_stat, p_val = stats.ttest_1samp(tcav_scores_mag, 0.5)
            
            print("\n--- RÉSULTATS FINAUX ---")
            print(f"Score TCAV Moyen : {avg_score:.4f} ± {std_score:.4f}")
            print(f"P-value : {p_val:.5e}")
            
            if p_val < 0.05:
                interpretation = "POSITIF" if avg_score > 0.5 else "NÉGATIF"
                print(f"✅ RÉSULTAT: Significatif ({interpretation})")
            else:
                print("❌ RÉSULTAT: Non significatif (Bruit)")
        else:
            print("❌ Pas assez de scores calculés.")

    finally:
        # Nettoyage final pour ne pas laisser le GPU encombré
        wrapper.clear_hooks()
        del wrapper, model
        torch.cuda.empty_cache()
        gc.collect()
        print("--- Mémoire GPU libérée ---")

# --- EXECUTION ---
if __name__ == "__main__":
    # Index ImageNet pour 'Zebra' est 340
    # Assurez-vous que les chemins sont corrects
    run_tcav_analysis(
        '/kaggle/input/data-tcav/data/concepts/stripes', 
        '/kaggle/input/data-tcav/data/target/zebra', 
        '/kaggle/input/data-tcav/data/random', 
        340
    )

--- Chargement du modèle ResNet50 ---
Hook attaché à la couche: layer3
--- 1. Extraction des activations du Concept ---
--- 2. Extraction des activations Cible ---
--- 3. Boucle sur 15 ensembles aléatoires ---
Run 0: Score Mag = 0.9569
Run 1: Score Mag = 0.9839
Run 2: Score Mag = 0.9801
Run 3: Score Mag = 0.6801
Run 4: Score Mag = 0.9581
Run 5: Score Mag = 0.8903
Run 6: Score Mag = 0.5538
Run 7: Score Mag = 0.7112
Run 8: Score Mag = 0.8475
Run 9: Score Mag = 0.6830
Run 10: Score Mag = 0.5295
Run 11: Score Mag = 0.6768
Run 12: Score Mag = 0.7372
Run 13: Score Mag = 0.7538
Run 14: Score Mag = 0.6618

--- RÉSULTATS FINAUX ---
Score TCAV Moyen : 0.7736 ± 0.1474
P-value : 6.81530e-06
✅ RÉSULTAT: Significatif (POSITIF)
--- Mémoire GPU libérée ---
